In [ ]:
import os
import pandas as pd

# Clean files path
data_dir = r"D:\CadetX\HeavySuppliersWarehouseDatasets\processed_data"

# 1. Load files
products = pd.read_csv(os.path.join(data_dir, "products.csv"))
inventory = pd.read_csv(os.path.join(data_dir, "inventory_master.csv"))
branches = pd.read_csv(os.path.join(data_dir, "branches.csv"))

# 2. Merge Inventory + Products (Key: product_id)
inv_prod = pd.merge(inventory, products, on="product_id", how="left")

# 3. Merge with Branches / Warehouse (Key: branch_id)
dim_product_inventory = pd.merge(
    inv_prod, branches, on="branch_id", how="left"
)

# 4. Check shape and columns
print("Merged Shape:", dim_product_inventory.shape)
print("Merged Columns:", dim_product_inventory.columns.tolist())

# Save your merged portion
dim_product_inventory.to_csv(
    os.path.join(data_dir, "dim_product_inventory.csv"), index=False
)
print("✅ Chandrashekhar's merged dimension ready!")


Merged Shape: (180, 43)
Merged Columns: ['product_id', 'branch_id', 'opening_stock', 'reorder_level_x', 'safety_stock_x', 'max_stock', 'current_stock', 'warehouse_bin', 'product_name', 'category', 'machine_type', 'brand', 'model_compatibility', 'unit_cost', 'unit_price', 'margin_percentage', 'gst_rate', 'weight_kg', 'dimensions_cm', 'material_type', 'warranty_months', 'reorder_level_y', 'safety_stock_y', 'max_stock_level', 'lead_time_days', 'criticality_level', 'usage_frequency', 'uom', 'last_purchase_price', 'last_purchase_date', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index', 'warehouse_capacity_sqft']
✅ Chandrashekhar's merged dimension ready!


In [1]:
import os
import pandas as pd

data_dir = r"D:\CadetX\HeavySuppliersWarehouseDatasets\processed_data"

# 1. Load files
products = pd.read_csv(os.path.join(data_dir, "products.csv"))
inventory = pd.read_csv(os.path.join(data_dir, "inventory_master.csv"))
branches = pd.read_csv(os.path.join(data_dir, "branches.csv"))

# 2. Duplicate columns ko handle karte hue Merge Karein
# Products me se duplicate stock fields drop kar dete hain taaki _x aur _y na bane
products_clean = products.drop(
    columns=["reorder_level", "safety_stock"], errors="ignore"
)

# Merge Inventory + Products
inv_prod = pd.merge(inventory, products_clean, on="product_id", how="left")

# Merge with Branches
dim_product_inventory = pd.merge(
    inv_prod, branches, on="branch_id", how="left"
)

# Save final dimension table
dim_product_inventory.to_csv(
    os.path.join(data_dir, "dim_product_inventory.csv"), index=False
)

print("✅ Clean Dimension Table Created Successfully!")
print("Final Shape:", dim_product_inventory.shape)

✅ Clean Dimension Table Created Successfully!
Final Shape: (180, 41)


In [2]:
dim_product_inventory

,product_id,branch_id,opening_stock,reorder_level,safety_stock,max_stock,current_stock,warehouse_bin,product_name,category,...,region,warehouse_type,warehouse_capacity,service_center_available,manager_id,total_employees,avg_monthly_revenue,monthly_operational_cost,market_demand_index,warehouse_capacity_sqft
0,P001,DEL001,196,59,46,340,115928,C15,Hydraulic Pump HP-300,Hydraulic,...,North,Central,45230 sqft,Yes,9593,92,5234890,3189000,9,45230
1,P001,PUN001,198,70,43,349,109389,D16,Hydraulic Pump HP-300,Hydraulic,...,West,Regional,38750 sqft,Yes,6114,74,4389120,2654000,8,38750
2,P001,CHN001,187,62,44,311,99373,A11,Hydraulic Pump HP-300,Hydraulic,...,South,Regional,41890 sqft,Yes,7922,81,4567780,2739000,8,41890
3,P001,HYD001,226,71,65,417,108772,A10,Hydraulic Pump HP-300,Hydraulic,...,South,Branch,36210 sqft,No,4390,57,3102455,1887000,7,36210
4,P001,KOL001,263,94,58,475,103808,F03,Hydraulic Pump HP-300,Hydraulic,...,East,Branch,29880 sqft,No,5981,49,2894770,1743000,6,29880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,P030,PUN001,147,52,35,283,112388,A05,Fuel Pump FP-350,Engine,...,West,Regional,38750 sqft,Yes,6114,74,4389120,2654000,8,38750
176,P030,CHN001,125,46,28,246,105447,F19,Fuel Pump FP-350,Engine,...,South,Regional,41890 sqft,Yes,7922,81,4567780,2739000,8,41890
177,P030,HYD001,162,53,40,268,104611,D14,Fuel Pump FP-350,Engine,...,South,Branch,36210 sqft,No,4390,57,3102455,1887000,7,36210
178,P030,KOL001,199,73,43,365,103465,C10,Fuel Pump FP-350,Engine,...,East,Branch,29880 sqft,No,5981,49,2894770,1743000,6,29880


In [1]:
import os
import pandas as pd

# File path
data_dir = r"D:\CadetX\HeavySuppliersWarehouseDatasets\processed_data"

# 1. Load Sales, Customer, Supplier & Invoice Files
so_lines = pd.read_csv(os.path.join(data_dir, "sales_orders_lines.csv"))
so_header = pd.read_csv(os.path.join(data_dir, "sales_orders_header.csv"))
customers = pd.read_csv(os.path.join(data_dir, "customers.csv"))
suppliers = pd.read_csv(os.path.join(data_dir, "suppliers.csv"))
invoices = pd.read_csv(os.path.join(data_dir, "invoices.csv"))
po_header = pd.read_csv(os.path.join(data_dir, "purchase_orders_header.csv"))

# 2. Load Chandrashekhar's Merged Dimension (Products + Inventory + Branches)
dim_prod_inv = pd.read_csv(
    os.path.join(data_dir, "dim_product_inventory.csv")
)

# ---------------------------------------------------------
# STEP A: Salim's Task — Merge Sales, Customers & Suppliers
# ---------------------------------------------------------
# Merge Sales Lines with Header on 'so_id'
sales_base = pd.merge(so_lines, so_header, on="so_id", how="left")

# Merge Sales with Customers on 'customer_id'
sales_cust = pd.merge(sales_base, customers, on="customer_id", how="left")

# Merge Suppliers info via Purchase Orders Header (if needed)
po_supp = pd.merge(
    po_header[["po_id", "supplier_id"]], suppliers, on="supplier_id", how="left"
)

print("✅ Sales, Customer, and Supplier merged successfully!")

# ---------------------------------------------------------
# STEP B: Combine All Components into Master 'merged_dataset.csv'
# ---------------------------------------------------------
# Merge Sales + Customer data with Product-Inventory-Branch Dimension
master_df = pd.merge(
    sales_cust, dim_prod_inv, on=["product_id", "branch_id"], how="left"
)

# Merge Invoices safely using 'so_id'
master_df = pd.merge(
    master_df,
    invoices[
        [
            "so_id",
            "invoice_id",
            "invoice_key",
            "invoice_date",
            "payment_status",
            "grand_total",
        ]
    ],
    on="so_id",
    how="left",
)

# Save Master Dataset
output_path = os.path.join(data_dir, "merged_dataset.csv")
master_df.to_csv(output_path, index=False)

print("\n🎉 ALL 6 TABLES SUCCESSFULLY MERGED!")
print("Final Merged Dataset Shape:", master_df.shape)
print(f"File Saved at: {output_path}")

✅ Sales, Customer, and Supplier merged successfully!


KeyError: 'branch_id'

In [2]:
import os
import pandas as pd

# File path
data_dir = r"D:\CadetX\HeavySuppliersWarehouseDatasets\processed_data"

# 1. Load Sales, Customer, Supplier & Invoice Files
so_lines = pd.read_csv(os.path.join(data_dir, "sales_orders_lines.csv"))
so_header = pd.read_csv(os.path.join(data_dir, "sales_orders_header.csv"))
customers = pd.read_csv(os.path.join(data_dir, "customers.csv"))
invoices = pd.read_csv(os.path.join(data_dir, "invoices.csv"))

# 2. Load Chandrashekhar's Merged Dimension (Products + Inventory + Branches)
dim_prod_inv = pd.read_csv(
    os.path.join(data_dir, "dim_product_inventory.csv")
)

# ---------------------------------------------------------
# STEP A: Merge Sales and Customers
# ---------------------------------------------------------
sales_base = pd.merge(so_lines, so_header, on="so_id", how="left")
sales_cust = pd.merge(sales_base, customers, on="customer_id", how="left")

# ---------------------------------------------------------
# STEP B: Join with Product-Inventory Dimension safely
# ---------------------------------------------------------
# Check join keys between Sales Base and Dimension Table
if "branch_id" in sales_cust.columns:
    join_keys = ["product_id", "branch_id"]
else:
    join_keys = ["product_id"]

master_df = pd.merge(sales_cust, dim_prod_inv, on=join_keys, how="left")

# ---------------------------------------------------------
# STEP C: Merge Invoices safely
# ---------------------------------------------------------
invoice_cols = [
    col
    for col in [
        "so_id",
        "invoice_id",
        "invoice_key",
        "invoice_date",
        "payment_status",
        "grand_total",
    ]
    if col in invoices.columns
]
master_df = pd.merge(master_df, invoices[invoice_cols], on="so_id", how="left")

# Save Master Dataset
output_path = os.path.join(data_dir, "merged_dataset.csv")
master_df.to_csv(output_path, index=False)

print("🎉 ALL TABLES SUCCESSFULLY MERGED!")
print("Final Merged Dataset Shape:", master_df.shape)

🎉 ALL TABLES SUCCESSFULLY MERGED!
Final Merged Dataset Shape: (782412, 78)
